# BruSafe · Notebook 02 — Module M1: expression, lactose autoinduction arm (24 h)

*Section numbering follows the CAPSID framework document; this notebook implements Sections 2.7–2.10 and 2.13–2.19.*

**iGEM Tec-Chihuahua 2026 · CAPSID framework**

This notebook is the companion to `BruSafe_01_M1_expression_IPTG.ipynb`. The structure and the validation scheme are the same, with two substantive changes:

1. **Allolactose replaces IPTG as the inducer.** Allolactose is the natural effector of LacI, but it cannot be set directly: it is produced from lactose by β-galactosidase and destroyed by the same enzyme. A single transport term therefore becomes a subsystem with feedback.
2. **The horizon extends to 24 h instead of 5 h.** The reason is given below and in §3.

## Why the horizon had to move

Under IPTG the inducer is present at full concentration within minutes. Under lactose it is not:
LacY must be expressed to import lactose, β-galactosidase must be expressed to convert it, and both
are themselves under lac control, so induction bootstraps itself. Catabolite repression adds a
further delay, because little happens until glucose is exhausted.

At the calibrated parameters, **300 min captures only 1.3–2.3% of the protein present at 24 h**,
and half-maximal accumulation occurs at 650–780 min. A 300 min experiment under lactose induction
therefore does not measure a partly completed induction; it measures the lag.

## Workflow

Identical to notebook 01: literature priors → prior predictive → identifiability audit →
data (one swap point) → SMC calibration → recovery check → posterior predictive → held-out
validation → export for M2.

---
## 1 · Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from numba import njit
from scipy.integrate import solve_ivp
import pymc as pm, pytensor.tensor as pt, arviz as az
from pytensor.compile.ops import as_op

RNG = np.random.default_rng(2026)
plt.rcParams.update({"figure.dpi":110,"font.size":9,"axes.grid":True,"grid.alpha":.25,
                     "axes.spines.top":False,"axes.spines.right":False})
print("pymc", pm.__version__, "| arviz", az.__version__)

# ---- Figure style (team palette) -------------------------------------------
# The colours are sampled from the team wordmark. Figures refer to the names
# below instead of hex codes, so the whole set can be restyled from this cell.
# SF Pro is used where installed; matplotlib substitutes another font otherwise.
CREAM, PALE, STEEL = "#F4F3EF", "#CCD8E2", "#89A1C2"
SKY,   BLUE, MID   = "#4CA3DF", "#3E77E1", "#2C5BA5"
NAVY,  INK         = "#173779", "#122047"
from matplotlib.colors import LinearSegmentedColormap
BRAND_CMAP = LinearSegmentedColormap.from_list("tecchihuahua",
                                               [NAVY, MID, BLUE, SKY])
plt.rcParams.update({
    "font.family": ["SF Pro Text", "SF Pro Display", "SF Pro", "Helvetica Neue"],
    "mathtext.fontset": "custom", "mathtext.default": "regular",
    "mathtext.rm": "SF Pro Text", "mathtext.it": "SF Pro Text:italic",
    "mathtext.bf": "SF Pro Text:bold",
    "figure.facecolor": CREAM, "axes.facecolor": CREAM, "savefig.facecolor": CREAM,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.color": STEEL, "grid.alpha": 0.35,
    "legend.facecolor": CREAM, "legend.edgecolor": STEEL, "legend.framealpha": 0.9,
    "axes.prop_cycle": plt.cycler(color=[NAVY, BLUE, SKY, STEEL, MID, PALE]),
})


---
## 2 · The model

### 2.1 · State variables (ten, against five for IPTG)

| Symbol | Meaning | Units |
|---|---|---|
| `X` | biomass | gDCW L⁻¹ |
| `Glc` | extracellular glucose (catabolite repressor) | mM |
| `Le`, `Li` | extra-/intracellular lactose | mM |
| `Al` | intracellular **allolactose**, the actual LacI effector | mM |
| `B` | β-galactosidase (LacZ) | normalised 0–1 |
| `Yp` | lactose permease (LacY) | normalised 0–1 |
| `R̃` | T7 RNA polymerase | normalised 0–1 |
| `m̃` | CP transcript | normalised 0–1 |
| `P` | cumulative CP monomer | µM |

### 2.2 · Equations

Capacity and growth, unchanged from the IPTG model:

$$\psi=\max\!\left(0,1-\tfrac{P/P_{ref}}{\phi_{crit}}\right),\qquad
\mu=\mu_{max}\max\!\left(0,1-\tfrac{X}{X_{max}}\right)\psi,\qquad \frac{dX}{dt}=\mu X$$

Catabolite repression and inducer exclusion, both driven by glucose:

$$\mathcal{C}(Glc)=\frac{1}{1+Glc/K_{cat}},\qquad
\xi(Glc)=\frac{1}{1+Glc/K_{ie}},\qquad
\frac{d\,Glc}{dt}=-q_{Glc}X\frac{Glc}{K_{upt}+Glc}$$

Lactose uptake through LacY, followed by the two reactions of β-galactosidase on lactose
(hydrolysis and transgalactosylation):

$$v_{up}=V_Y\,Y_p\frac{Le}{K_Y+Le}\,\xi,\qquad
s_L=k_{cat}B\frac{Li}{K_{ML}+Li},\qquad
v_{tg}=f_{tg}s_L,\quad v_{hyd}=(1-f_{tg})s_L$$

$$\frac{dLi}{dt}=v_{up}-v_{hyd}-v_{tg}-\mu Li$$

Allolactose is produced by transgalactosylation and destroyed by the same enzyme:

$$v_{ah}=k_{cat}B\frac{Al}{K_{MA}+Al},\qquad
\frac{dAl}{dt}=v_{tg}-v_{ah}-\mu Al$$

De-repression by allolactose, followed by the two branches of the response:

$$f(Al)=\varepsilon+(1-\varepsilon)\frac{Al^{h}}{K_A^{h}+Al^{h}}$$

$$\frac{dB}{dt}=k_{lac}(\mathcal{C}f-B),\qquad
\frac{dY_p}{dt}=k_{lac}(\mathcal{C}f-Y_p),\qquad
\frac{d\tilde R}{dt}=k_R(f-\tilde R)$$

$$\frac{d\tilde m}{dt}=k_m(\tilde R-\tilde m),\qquad
\frac{dP}{dt}=G\,\tilde m\,\psi-(\mu+k_{deg})P$$

### 2.3 · Four modelling decisions

**(a) β-galactosidase degrades allolactose as fast as it forms it.** Huber and co-workers report
a $K_M$ of 1.2 mM and a V of 49.6 U mg⁻¹ for allolactose, so it is at least as good a
substrate as lactose. Omitting $v_{ah}$ would let allolactose accumulate without bound, a common
error in simplified lac models. The term is a negative feedback that substantially damps the
positive one.

**(b) Only the chromosomal *lac* branch is catabolite-repressed; lacUV5 is not.** The UV5 mutation
was isolated for catabolite insensitivity, so $\mathcal{C}$ multiplies $B$ and $Y_p$ but not
$\tilde R$. Glucose still gates the whole system indirectly, through LacY and hence allolactose
supply. This coupling exists because uptake is the bottleneck.

**(c) A *lacY*⁻ strain is no longer an option.** Notebook 01 recommended Tuner(DE3) to remove the
LacY-mediated positive feedback. Under lactose, LacY is the uptake route, so the feedback is
structural. §4 tests its consequences.

**(d) Protein degradation is no longer negligible.** Over 5 h, dilution by growth dominates and
$k_{deg}$ can be set to zero. Over 24 h, with growth arrested for most of the run, proteolysis is
the dominant loss term for soluble protein, and $k_{deg}$ is therefore fitted.

In [ ]:
P_REF = 18000.0    # uM total cytoplasmic protein, CP-mass equivalents  [M1.4]
M_CP  = 13.7       # kDa
# ---- Biomass and protein in physical units ---------------------------------
# OD600 and SDS-PAGE band volume are instrument readings, not concentrations,
# and band volumes in particular cannot be compared between gels. The model
# therefore works with biomass concentration (gDCW/L) and coat-protein
# concentration (mg/L); the raw readings enter only through the explicit
# calibrations defined below.
K_DCW = 0.40      # gDCW/L per OD600 unit (E. coli in rich medium: 0.3-0.5);
                  # should be calibrated for the strain in use.
F_DRY = 0.25      # dry mass / wet pellet mass; multiply wet weight by this to get gDCW/L.
od_to_gdcw  = lambda od:  np.asarray(od, float)*K_DCW
wet_to_gdcw = lambda gww: np.asarray(gww, float)*F_DRY
uM_to_mgL   = lambda c:   np.asarray(c, float)*M_CP/1000.0    # uM CP -> mg/L
mgL_to_uM   = lambda c:   np.asarray(c, float)*1000.0/M_CP
# ----------------------------------------------------------------------------

PHI_CYT = 1.8e-3   # gCP per gDCW per uM CP (cytoplasmic volume factor)

FIT = ["mu_max","phi_crit","k_deg","K_cat","V_Y","k_cat","K_A","h","eps",
       "k_lac","k_slow","k_fast","G"]
FIXED = dict(Xmax=3.20,   # gDCW/L (= 8 OD600 x K_DCW)
             q_Glc=0.020, K_upt=0.10, K_ie=0.5,
             K_Y=0.40, f_tg=0.35, K_ML=1.5, K_MA=1.2)

# full parameter order consumed by the RHS
PORDER = ["mu_max","Xmax","phi_crit","k_deg","q_Glc","K_upt","K_ie","K_cat",
          "V_Y","K_Y","k_cat","f_tg","K_ML","K_MA","K_A","h","eps",
          "k_lac","k_slow","k_fast","G"]

@njit(fastmath=True)
def _rhs(t, Yv, p, Le0):
    n = Le0.size; out = np.empty(10*n)
    mu_max,Xmax,phi_crit,k_deg = p[0],p[1],p[2],p[3]
    q_Glc,K_upt,K_ie,K_cat     = p[4],p[5],p[6],p[7]
    V_Y,K_Y,k_cat,f_tg         = p[8],p[9],p[10],p[11]
    K_ML,K_MA,K_A,h,eps        = p[12],p[13],p[14],p[15],p[16]
    k_lac,k_R,k_m,G            = p[17],p[18],p[19],p[20]
    KAh = K_A**h
    for i in range(n):
        X=Yv[i]; Glc=Yv[n+i]; Le=Yv[2*n+i]; Li=Yv[3*n+i]; Al=Yv[4*n+i]
        B=Yv[5*n+i]; Yp=Yv[6*n+i]; R=Yv[7*n+i]; m=Yv[8*n+i]; P=Yv[9*n+i]
        if Glc<0.0: Glc=0.0
        if Le<0.0:  Le=0.0
        if Li<0.0:  Li=0.0
        if Al<0.0:  Al=0.0
        psi = 1.0-(P/P_REF)/phi_crit
        if psi<0.0: psi=0.0
        gx = 1.0-X/Xmax
        if gx<0.0: gx=0.0
        mu = mu_max*gx*psi                       # [M1.1][M1.2]
        Cc  = 1.0/(1.0+Glc/K_cat)                # [M1L.5] CRP-cAMP catabolite repression
        xie = 1.0/(1.0+Glc/K_ie)                 # [M1L.6] inducer exclusion
        v_up  = V_Y*Yp*(Le/(K_Y+Le))*xie         # [M1L.1] LacY uptake
        sL    = k_cat*B*(Li/(K_ML+Li))           # [M1L.2] beta-gal on lactose
        v_tg  = f_tg*sL                          # [M1L.3] transgalactosylation -> allolactose
        v_hyd = (1.0-f_tg)*sL                    # [M1L.3] hydrolysis
        v_ah  = k_cat*B*(Al/(K_MA+Al))           # [M1L.4] beta-gal ON allolactose
        f = eps + (1.0-eps)*Al**h/(KAh+Al**h)    # [M1L.9] LacI de-repression
        out[i]     = mu*X
        out[n+i]   = -q_Glc*X*(Glc/(K_upt+Glc))  # [M1L.7] glucose consumption
        out[2*n+i] = -v_up*PHI_CYT*X          # X is gDCW/L, so no OD conversion here
        out[3*n+i] = v_up - v_hyd - v_tg - mu*Li
        out[4*n+i] = v_tg - v_ah - mu*Al
        out[5*n+i] = k_lac*(Cc*f - B)            # [M1L.10] LacZ, catabolite-repressed
        out[6*n+i] = k_lac*(Cc*f - Yp)           # [M1L.10] LacY, catabolite-repressed
        out[7*n+i] = k_R*(f - R)                 # [M1L.11] lacUV5 -> T7 RNAP, NOT repressed
        out[8*n+i] = k_m*(R - m)                 # [M1.12]
        out[9*n+i] = G*m*psi - (mu+k_deg)*P      # [M1.16][M1L.12] proteolysis at long horizon
    return out

def theta_full(d):
    return np.array([d[k] for k in PORDER], float)

def simulate(theta, Le0, t_eval, X0=0.02, Glc0=2.5):
    p=np.asarray(theta,float); Le0=np.asarray(Le0,float); n=Le0.size
    te=np.asarray(t_eval,float)
    Y0=np.concatenate([np.full(n,X0),np.full(n,Glc0),Le0]+[np.zeros(n)]*7)
    s=solve_ivp(_rhs,(0.0,float(te[-1])),Y0,t_eval=te,args=(p,Le0),
                method="LSODA",rtol=1e-6,atol=1e-9)
    if (not s.success) or s.y.shape[1]!=te.size: return None
    Y=s.y
    return {"X":Y[0:n],"Glc":Y[n:2*n],"Le":Y[2*n:3*n],"Li":Y[3*n:4*n],"Al":Y[4*n:5*n],
            "B":Y[5*n:6*n],"Yp":Y[6*n:7*n],"R":Y[7*n:8*n],"m":Y[8*n:9*n],"P":Y[9*n:10*n]}

TRUE = dict(mu_max=0.021, phi_crit=0.25, k_deg=5e-4, K_cat=0.06, V_Y=0.32,
            k_cat=3.0, K_A=0.15, h=2.0, eps=0.005, k_lac=0.05,
            k_slow=0.07, k_fast=0.30, G=95.0, **FIXED)
_th0 = theta_full(TRUE)
simulate(_th0, np.array([5.5]), np.array([0.,60.]))   # JIT warm-up
import time; _t=time.time()
for _ in range(100): simulate(_th0, np.array([0.5,1.5,5.5,15.]), np.linspace(0,1440,10))
print(f"forward model: {(time.time()-_t)/100*1000:.1f} ms per evaluation")

### 2.4 · Term-by-term provenance

Tags `[M1L.x]` are new to this arm; tags `[M1.x]` carry over unchanged from notebook 01. The
evidence classes are as before: *Direct* (the cited work reports this form or value), *Adapted*
(the mechanism is established, the form is ours), *Assumption* (no literature claim; justified by
argument) and *Reparam.* (a mathematical restatement).

In [ ]:
CITES = pd.DataFrame([
 ("M1L.1","V_Y*Yp*Le/(K_Y+Le)*xi","Lactose enters only through LacY; K_Y ~ 0.4 mM",
  "Direct","Jacob & Monod 1961 JMB 3:318; Marbach & Bettenbrock 2012 J Biotechnol 157:82"),
 ("M1L.2","k_cat*B*Li/(K_ML+Li)","beta-galactosidase acts on intracellular lactose",
  "Direct","Juers, Matthews & Huber 2012 Protein Sci 21:1792"),
 ("M1L.3","f_tg split: v_tg vs v_hyd","beta-gal partitions lactose between hydrolysis and transgalactosylation to allolactose",
  "Direct","Huber, Kurz & Wallenfels 1976 Biochemistry 15:1994; Wheatley et al. 2013 JBC 288:12993"),
 ("M1L.4","v_ah = k_cat*B*Al/(K_MA+Al)","Allolactose is ITSELF a beta-gal substrate: K_M 1.2 mM, V 49.6 U/mg -- as good as lactose or better",
  "Direct","Huber, Wallenfels & Kurz 1975 Can J Biochem 53:1035"),
 ("M1L.5","C = 1/(1+Glc/K_cat)","Glucose lowers cAMP-CRP and represses the chromosomal lac promoter",
  "Adapted","Gorke & Stulke 2008 Nat Rev Microbiol 6:613; Deutscher 2008 Curr Opin Microbiol 11:87"),
 ("M1L.6","xi = 1/(1+Glc/K_ie)","Inducer exclusion: PTS-mediated inhibition of LacY by glucose",
  "Adapted","Gorke & Stulke 2008. Hyperbolic FORM is ours"),
 ("M1L.7","glucose consumed first","Diauxic sequencing is the basis of lactose autoinduction media",
  "Direct","Studier 2005 Protein Expr Purif 41:207"),
 ("M1L.8","C does NOT multiply dR/dt","lacUV5 was isolated as a catabolite-INSENSITIVE promoter mutant",
  "Direct","Silverstone, Arditti & Magasanik 1970 PNAS 66:773"),
 ("M1L.9","f(Al), Hill in ALLOLACTOSE","Allolactose, not lactose, is the physiological LacI inducer",
  "Direct","Jobe & Bourgeois 1972 JMB 69:397"),
 ("M1L.10","B and Yp share k_lac","LacZ and LacY are co-transcribed from one operon, so one time constant",
  "Adapted","Jacob & Monod 1961. Equal amplitude is a simplification -- LacY is sub-stoichiometric"),
 ("M1L.11","positive feedback loop","Al -> derepress -> more LacY/LacZ -> more Al. Structural, not removable",
  "Direct","Novick & Weiner 1957 PNAS 43:553; Ozbudak et al. 2004 Nature 427:737"),
 ("M1L.12","k_deg on P","Proteolysis dominates over dilution once growth arrests, i.e. most of a 24 h run",
  "Assumption","Rate is protein- and strain-specific; no generic literature value applies"),
 ("M1L.13","Glc0 = 2.5 mM default","Autoinduction media carry glucose to delay induction until it is spent",
  "Direct","Studier 2005 Protein Expr Purif 41:207"),
], columns=["tag","term","basis","evidence","reference"])
pd.set_option("display.max_colwidth",110)
print(CITES.evidence.value_counts().to_string(),"\n")
CITES

> **Caveat on M1L.10.** LacZ and LacY are co-transcribed, so one shared time constant is
> defensible. They are not produced in equal amounts, however: translational coupling makes LacY
> sub-stoichiometric to LacZ. Modelling both with the same amplitude folds that ratio into `V_Y`
> and `k_cat`. This is acceptable for fitting, but neither parameter should then be quoted as a
> physical rate.

> **Caveat on M1L.12.** No literature value exists for this term, and none is imposed.
> Proteolytic turnover depends on the protein, the strain and the growth phase. The term is
> fitted with a weak prior, and it is identifiable only because the horizon is long.

---
## 3 · Why a 300 min horizon is insufficient

In [ ]:
DOSES = np.array([0.5, 1.5, 5.5, 15.0])          # mM lactose (0.02% - 0.5% w/v)
TOBS  = np.array([0.,60.,120.,240.,360.,480.,720.,960.,1200.,1440.])
t_fine = np.linspace(0, 1440, 241)

o = simulate(_th0, DOSES, t_fine)
i300 = np.argmin(np.abs(t_fine-300))
frac = o["P"][:,i300]/o["P"][:,-1]
t50  = np.array([t_fine[np.argmax(o["P"][j] >= 0.5*o["P"][j,-1])] for j in range(len(DOSES))])
print("Fraction of 24 h protein captured by 300 min:", np.round(frac,3))
print("Time to half-maximal protein (min):        ", t50.astype(int))
print("Glucose exhausted at (min):                ",
      int(t_fine[np.argmax(o["Glc"][2] < 0.01)]))

fig, ax = plt.subplots(1,3, figsize=(12.5,3.4))
for j,d in enumerate(DOSES):
    c = BRAND_CMAP(j/3)
    ax[0].plot(t_fine, o["P"][j], color=c, lw=1.8, label=f"{d:g} mM")
    ax[1].plot(t_fine, o["Al"][j]*1e3, color=c, lw=1.8)
    ax[2].plot(t_fine, o["Yp"][j], color=c, lw=1.8)
ax[0].axvline(300, color=INK, ls="--", lw=1.4)
ax[0].text(320, o["P"].max()*0.12, "old horizon\n(300 min)", fontsize=7, color=INK)
ax[0].set(xlabel="time (min)", ylabel="CP ($\\mu$M)", title="Protein: the lag is the story")
ax[0].legend(fontsize=7, title="lactose")
ax[1].plot(t_fine, o["Glc"][0]/o["Glc"][0].max()*o["Al"].max()*1e3, ls=":", color=INK, lw=1.4)
ax[1].text(60, o["Al"].max()*1e3*0.85, "glucose (scaled)", fontsize=6.5)
ax[1].set(xlabel="time (min)", ylabel="allolactose ($\\mu$M)", title="Effector must be built first")
ax[2].set(xlabel="time (min)", ylabel="LacY (normalised)", title="Uptake bootstraps itself")
plt.tight_layout(); plt.show()

The middle panel shows that allolactose does not appear when lactose is added. It appears when
glucose runs out, because until then catabolite repression and inducer exclusion keep LacY off and
lactose out of the cell. The effector has to be made by the operon it induces.

This is the reason for extending the horizon. A 300 min experiment under lactose induction
captures the lag instead of a partly completed induction, and would return an almost flat
dose-response with strongly biased parameters.

---
## 4 · Identifiability audit

### 4.1 · The k_R ↔ k_m symmetry persists

The R̃ → m̃ cascade is unchanged from the IPTG model, so its transfer function is still exactly
symmetric under exchange of the two rate constants. The same ordered-pair fix applies: `k_slow`
and a ratio are fitted, and the unordered pair is reported.

`k_lac` belongs to a different branch (the feedback loop) and is not exchangeable with them.

### 4.2 · New degeneracies introduced by the allolactose subsystem

In [ ]:
# demonstrate the surviving cascade symmetry
ta = dict(TRUE); tb = dict(TRUE)
tb["k_slow"], tb["k_fast"] = TRUE["k_fast"], TRUE["k_slow"]
a = simulate(theta_full(ta), DOSES, t_fine); b = simulate(theta_full(tb), DOSES, t_fine)
print(f"k_slow<->k_fast :  max|dP|/max(P) = {np.abs(a['P']-b['P']).max()/a['P'].max():.2e}  -> exchangeable")

# V_Y and k_cat: both scale allolactose supply -- check how close to degenerate
tc = dict(TRUE); tc["V_Y"] = TRUE["V_Y"]*1.6; tc["k_cat"] = TRUE["k_cat"]/1.6
c = simulate(theta_full(tc), DOSES, t_fine)
print(f"V_Y*1.6 & k_cat/1.6:  max|dP|/max(P) = {np.abs(a['P']-c['P']).max()/a['P'].max():.2e}"
      "  -> NOT degenerate: both are separately identifiable")

# f_tg is fixed for parsimony (tested below: it is not degenerate with k_cat)
td = dict(TRUE); td["f_tg"] = 0.5; td["k_cat"] = TRUE["k_cat"]*0.35/0.5
d_ = simulate(theta_full(td), DOSES, t_fine)
print(f"f_tg & k_cat co-scaled: max|dP|/max(P) = {np.abs(a['P']-d_['P']).max()/a['P'].max():.2e}"
      "  -> also NOT degenerate (k_cat drives hydrolysis and v_ah too);")
print("     f_tg is fixed for PARSIMONY, not because it is unidentifiable.")

| Group | Tested result | Resolution |
|---|---|---|
| `k_R ↔ k_m` | **Exact symmetry**, ~10⁻⁶ relative difference | Ordered pair; report the unordered pair |
| `V_Y` × `k_cat` | **Not degenerate**: co-scaling changes the trajectory by ~70% | Both fitted |
| `f_tg` × `k_cat` | **Not degenerate**: `k_cat` also drives hydrolysis and `v_ah` | `f_tg` fixed for *parsimony*, not identifiability |
| `K_cat` vs `K_ie` | Both gate on glucose through one variable | `K_ie` fixed; `K_cat` fitted |
| `k_deg` | Identifiable **only** because the horizon is long | Would be unidentifiable at 300 min |

The `f_tg`/`k_cat` pair was expected to be nearly degenerate and is not, because `k_cat` also sets
hydrolysis and allolactose destruction: one rate constant controls three fluxes. The test takes
three lines and changed the conclusion, so this result was tested and not assumed.

**The main ridge lies elsewhere and is severe.** After fitting, the strongest posterior
correlations are between the basal leak `eps` and the gain `G` (r ≈ −0.98) and between `k_cat`
and `G` (r ≈ +0.87). A leakier promoter with a smaller gain produces almost the same protein curve.
The uninduced control arm breaks this trade-off, which is why §12 lists it as required.

### 4.3 · Does the positive feedback produce bistability?

Under IPTG, the *lacY*⁺ positive feedback makes induction bimodal at sub-saturating inducer,
which is why notebook 01 recommended a *lacY*⁻ strain. Under lactose that option is unavailable.

The allolactose system, however, carries a negative feedback that gratuitous inducers lack:
β-galactosidase destroys allolactose (term M1L.4). Classic bistability experiments used TMG and
other non-metabolisable analogues precisely because they are not consumed. Whether bistability
survives here is a quantitative question, and it is tested below.

In [ ]:
def run_from(Le0, Y_init, B_init, t_end=1440.0, npts=121):
    Le0 = np.asarray(Le0,float); n = Le0.size; te = np.linspace(0,t_end,npts)
    Y0 = np.concatenate([np.full(n,0.05), np.full(n,2.5), Le0, np.zeros(n), np.zeros(n),
                         np.full(n,B_init), np.full(n,Y_init), np.zeros(n), np.zeros(n), np.zeros(n)])
    s = solve_ivp(_rhs,(0,t_end),Y0,t_eval=te,args=(_th0,Le0),method="LSODA",rtol=1e-6,atol=1e-9)
    return te, s.y[9*n:10*n]

scan = np.array([0.02,0.05,0.10,0.20,0.40,0.80,1.6,3.2,6.4,12.8])
_, P_naive = run_from(scan, 0.0, 0.0)     # uninduced inoculum
_, P_pre   = run_from(scan, 1.0, 1.0)     # fully pre-induced inoculum
gap = np.abs(P_pre[:,-1]-P_naive[:,-1])/np.maximum(P_naive[:,-1],1e-9)

fig, ax = plt.subplots(1,2, figsize=(9,3.3))
ax[0].plot(scan, P_naive[:,-1], "o-", color=BLUE, lw=1.8, label="naive inoculum")
ax[0].plot(scan, P_pre[:,-1], "s--", color=INK, lw=1.6, label="pre-induced inoculum")
ax[0].set(xscale="log", xlabel="lactose (mM)", ylabel="CP at 24 h ($\\mu$M)",
          title="Hysteresis test"); ax[0].legend(fontsize=7)
ax[1].plot(scan, 100*gap, "o-", color=MID, lw=1.8)
ax[1].set(xscale="log", xlabel="lactose (mM)", ylabel="| relative gap | (%)",
          title="Path dependence")
plt.tight_layout(); plt.show()
print(f"max relative gap between histories: {100*gap.max():.2f}%")
print("Below ~1% -> single attractor at these parameters: a deterministic mean is meaningful.")

**Result at the nominal parameters: no hysteresis and a graded dose-response.** The two initial
histories converge to the same 24 h endpoint.

The mechanism is M1L.4. Allolactose is consumed by the enzyme whose synthesis its own accumulation
induces, and this negative loop damps the positive one enough to collapse the bistable region. For
a deterministic model this is a real, and somewhat counter-intuitive, advantage of lactose over
IPTG: the escape from bimodality that notebook 01 could obtain only with a special strain comes
without one.

This result should not be treated as settled. Bistability depends on the parameters, and the test
above uses nominal values. §11 repeats it with draws from the fitted posterior; if an appreciable
fraction of draws shows a gap, that fraction should be reported instead of a yes/no answer.

---
## 5 · Data

> ### ⇄ SWAP POINT: the only cell to change when real data arrive
>
> Required arrays, as in notebook 01:
> - `X_obs`: `(n_doses, n_times)`, biomass in gDCW L⁻¹ (OD₆₀₀ × `K_DCW`, or wet pellet weight × `F_DRY`)
> - `CP_obs`: `(n_doses, n_times)`, densitometry in **arbitrary units**
> - `GEL_ID`: length `n_doses`, the physical gel on which each condition was run
>
> **Two additions specific to this experiment:**
>
> 1. **Residual glucose and lactose** should be measured (enzymatic assay or HPLC) at 3–4 time
>    points. The time of glucose exhaustion pins `K_cat` and `q_Glc` almost on its own, and it is
>    the least expensive improvement in identifiability available here.
> 2. **Overnight sampling should be planned.** Half-maximal accumulation occurs near 11–13 h, so a
>    run started at 09:00 needs samples around 20:00 and 22:00.

In [ ]:
USE_SYNTHETIC = True          # set to False once real data are available

if USE_SYNTHETIC:
    truth = simulate(_th0, DOSES, TOBS)
    GEL_ID     = np.array([0,0,1,1])
    ALPHA_TRUE = np.array([1.0,0.78])          # standard-curve slope per gel
    X_obs  = truth["X"]*np.exp(RNG.normal(0,0.05,truth["X"].shape))          # gDCW/L
    CP_obs = (ALPHA_TRUE[GEL_ID][:,None]*uM_to_mgL(truth["P"])+0.05)*np.exp(
                 RNG.normal(0,0.12,truth["P"].shape))                        # mg/L
    print("SYNTHETIC data. Ground truth known -> recovery checkable in §8.")
else:
    # Expected columns: lactose_mM, time_min, gel_id, CP_mg_L, and EITHER
    # biomass_gdcw_L or wet_weight_g_L or OD600.
    df = pd.read_csv("VA_lactose_timecourse.csv")
    dr = np.sort(df.lactose_mM.unique()); tr = np.sort(df.time_min.unique())
    piv = lambda c: df.pivot_table(index="lactose_mM",columns="time_min",values=c).loc[dr,tr].values
    if   "biomass_gdcw_L" in df: X_obs = piv("biomass_gdcw_L")
    elif "wet_weight_g_L" in df: X_obs = wet_to_gdcw(piv("wet_weight_g_L"))
    else:                        X_obs = od_to_gdcw(piv("OD600"))
    CP_obs = piv("CP_mg_L")
    GEL_ID = df.groupby("lactose_mM").gel_id.first().loc[dr].values.astype(int)
    DOSES, TOBS = dr, tr
    print(f"Real data: {X_obs.shape[0]} doses x {X_obs.shape[1]} timepoints")

N_GEL = int(GEL_ID.max())+1; LOG_CP = np.log(CP_obs)
print("lactose (mM):", DOSES, "|", N_GEL, "gel(s) |", len(TOBS), "timepoints to", TOBS[-1], "min")
print(f"biomass {X_obs.min():.2f}-{X_obs.max():.2f} gDCW/L | CP {CP_obs.min():.2f}-{CP_obs.max():.1f} mg/L")

---
## 6 · Observation model

Unchanged from notebook 01: a per-gel scale `α_k` with gel 0 fixed at 1, log-normal densitometry
error, and a positive noise floor on biomass.

In [ ]:
@as_op(itypes=[pt.dvector], otypes=[pt.dmatrix, pt.dmatrix])
def forward_op(th_fit):
    d = dict(zip(FIT, th_fit)); d.update(FIXED)
    o = simulate(theta_full(d), DOSES, TOBS)
    if o is None:
        bad = np.full((DOSES.size,TOBS.size), 1e6); return bad, bad
    # second output: log of PREDICTED CP CONCENTRATION in mg/L
    return o["X"], np.log(o["P"]*M_CP/1000.0 + 0.05)

def build_model():
    with pm.Model() as M:
        mu_max = pm.TruncatedNormal("mu_max",0.021,0.005,lower=0.005,upper=0.045)
        phic   = pm.TruncatedNormal("phi_crit",0.25,0.08,lower=0.05,upper=0.50)
        k_deg  = pm.LogNormal("k_deg",np.log(5e-4),1.0)
        K_cat  = pm.LogNormal("K_cat",np.log(0.06),0.8)
        V_Y    = pm.LogNormal("V_Y",np.log(0.32),0.7)
        k_cat  = pm.LogNormal("k_cat",np.log(3.0),0.7)
        K_A    = pm.LogNormal("K_A",np.log(0.15),0.7)
        h      = pm.TruncatedNormal("h",2.0,0.5,lower=1.0,upper=4.0)
        eps    = pm.LogNormal("eps",np.log(0.005),0.8)
        k_lac  = pm.LogNormal("k_lac",np.log(0.05),0.6)
        G      = pm.LogNormal("G",np.log(95.0),0.8)
        k_slow = pm.LogNormal("k_slow",np.log(0.07),0.6)
        rho    = pm.Beta("rho",2.0,2.0)
        k_fast = pm.Deterministic("k_fast", k_slow/rho)
        if N_GEL > 1:
            la = pm.Normal("log_alpha_free",0.0,0.3,shape=N_GEL-1)
            log_alpha = pm.Deterministic("log_alpha", pt.concatenate([pt.zeros(1), la]))
        else:
            log_alpha = pm.Deterministic("log_alpha", pt.zeros(1))
        sig_X  = pm.HalfNormal("sig_X",0.08)    # relative error on gDCW/L
        sig_cp = pm.HalfNormal("sig_cp",0.20)   # relative error on mg/L
        th = pt.stack([mu_max,phic,k_deg,K_cat,V_Y,k_cat,K_A,h,eps,k_lac,k_slow,k_fast,G])
        X_m, logcp_m = forward_op(th)
        pm.Normal("X_lik", mu=X_m, sigma=sig_X*X_m+0.008, observed=X_obs)
        pm.Normal("cp_lik", mu=log_alpha[GEL_ID][:,None]+logcp_m, sigma=sig_cp, observed=LOG_CP)
    return M

model = build_model(); print(f"model built | {len(FIT)} fitted parameters, {len(FIXED)} fixed")

---
## 7 · Sampling

SMC is used as before: the forward model is a `scipy` solve behind a `pytensor` Op, so no gradients
are available and NUTS cannot be used. Thirteen parameters are fitted here, against eleven for the
IPTG arm, so runs are proportionally longer.

In [ ]:
FAST_MODE = False
DRAWS, CHAINS = (400, 4) if FAST_MODE else (2000, 4)
print(f"SMC: {DRAWS} draws x {CHAINS} chains"
      f"{'  [FAST_MODE -- re-run with FAST_MODE=False before reporting]' if FAST_MODE else ''}")
import time; _t0=time.time()
with model:
    idata = pm.sample_smc(draws=DRAWS, chains=CHAINS, random_seed=2026, progressbar=False)
print(f"done in {time.time()-_t0:.0f} s |", dict(idata.posterior.sizes))

---
## 8 · Diagnostics and recovery

In [ ]:
VARS = FIT + ["sig_X","sig_cp"]
summ = az.summary(idata, var_names=VARS); display(summ)
bad = summ[summ.r_hat > 1.01]
print(("\n!! r_hat>1.01: "+", ".join(bad.index)) if len(bad) else "\nAll r_hat <= 1.01")
low = summ[summ.ess_bulk < 200]
print(("!! ESS<200: "+", ".join(low.index)) if len(low) else "All ESS >= 200")

In [ ]:
if USE_SYNTHETIC:
    rows=[]
    for v in FIT:
        s_ = idata.posterior[v].values.ravel(); lo,hi = np.percentile(s_,[3,97])
        rows.append({"param":v,"truth":TRUE[v],"median":np.median(s_),
                     "lo94":lo,"hi94":hi,"recovered":bool(lo<=TRUE[v]<=hi)})
    rec = pd.DataFrame(rows); display(rec.round(5))
    print(f"\n{rec.recovered.sum()}/{len(rec)} recovered within the 94% credible interval.")
    print("~1 miss in 13 is expected by chance. Three or more -> diagnose before trusting real data.")

In [ ]:
def kde1d(x, grid):
    x=np.asarray(x); bw=1.06*np.std(x)*x.size**(-0.2)+1e-12
    z=(grid[:,None]-x[None,:])/bw
    return np.exp(-0.5*z**2).sum(1)/(x.size*bw*np.sqrt(2*np.pi))

SHOW = ["K_A","h","V_Y","k_cat","K_cat","k_deg","phi_crit","G"]
fig,axes = plt.subplots(2,4,figsize=(12,5.2))
for ax,v in zip(axes.ravel(), SHOW):
    po = idata.posterior[v].values.ravel()
    lo,hi = np.percentile(po,[0.5,99.5]); g=np.linspace(lo,hi,240)
    d_=kde1d(po,g); ax.fill_between(g,d_/d_.max(),color=BLUE,alpha=.55)
    ax.plot(np.percentile(po,[3,97]),[-.06,-.06],color=BLUE,lw=2.5,solid_capstyle="butt")
    if USE_SYNTHETIC: ax.axvline(TRUE[v],color=INK,ls="--",lw=1.6)
    ax.set_title(v,fontsize=9); ax.set_yticks([]); ax.set_ylim(-.14,1.15); ax.tick_params(labelsize=7)
fig.suptitle("Posterior marginals (dashed line = generating value)",fontsize=10)
plt.tight_layout(); plt.show()

PAIR=["eps","G","k_cat","V_Y"]
L=np.log(np.column_stack([idata.posterior[v].values.ravel() for v in PAIR])); n=len(PAIR)
fig,ax=plt.subplots(n,n,figsize=(7.2,7.0))
for i in range(n):
    for j in range(n):
        a_=ax[i,j]
        if i==j:
            g=np.linspace(L[:,i].min(),L[:,i].max(),200)
            a_.fill_between(g,kde1d(L[:,i],g),color=BLUE,alpha=.5); a_.set_yticks([])
        elif i>j: a_.plot(L[:,j],L[:,i],".",ms=1.2,alpha=.25,color=BLUE)
        else:
            r=np.corrcoef(L[:,j],L[:,i])[0,1]
            a_.text(.5,.5,f"r = {r:+.2f}",ha="center",va="center",transform=a_.transAxes,
                    fontsize=11 if abs(r)>.4 else 9, color=INK if abs(r)>.4 else STEEL)
            a_.set_xticks([]); a_.set_yticks([])
        if i==n-1: a_.set_xlabel(f"log {PAIR[j]}",fontsize=8)
        if j==0 and i!=0: a_.set_ylabel(f"log {PAIR[i]}",fontsize=8)
        a_.tick_params(labelsize=6)
fig.suptitle("Joint structure -- note the eps/G ridge: leak and gain trade off",fontsize=10)
plt.tight_layout(); plt.show()

---
## 9 · Posterior predictive

In [ ]:
def post_draws(idat, n=250, rng=RNG):
    f={v:idat.posterior[v].values.ravel() for v in FIT}
    i=rng.choice(f["G"].size, size=min(n,f["G"].size), replace=False)
    return [dict(zip(FIT,[f[k][ii] for k in FIT])) for ii in i]

TH_P = post_draws(idata,250); Pp,Xp=[],[]
for d in TH_P:
    dd=dict(d); dd.update(FIXED)
    o=simulate(theta_full(dd),DOSES,t_fine)
    if o is not None: Pp.append(o["P"]); Xp.append(o["X"])
Pp,Xp=np.array(Pp),np.array(Xp)
a_post=np.exp(idata.posterior["log_alpha"].values.reshape(-1,N_GEL)).mean(0)

fig,ax=plt.subplots(2,len(DOSES),figsize=(3.1*len(DOSES),5.4),sharex=True)
for j,d in enumerate(DOSES):
    q=np.percentile(uM_to_mgL(Pp[:,j,:])*a_post[GEL_ID[j]],[3,25,50,75,97],axis=0)
    ax[0,j].fill_between(t_fine,q[0],q[4],alpha=.15,color=BLUE)
    ax[0,j].fill_between(t_fine,q[1],q[3],alpha=.30,color=BLUE)
    ax[0,j].plot(t_fine,q[2],color=BLUE,lw=1.8)
    ax[0,j].plot(TOBS,CP_obs[j],"o",ms=5,color=INK,zorder=5)
    ax[0,j].set_title(f"{d:g} mM lactose",fontsize=9)
    qo=np.percentile(Xp[:,j,:],[3,50,97],axis=0)
    ax[1,j].fill_between(t_fine,qo[0],qo[2],alpha=.2,color=SKY)
    ax[1,j].plot(t_fine,qo[1],color=SKY,lw=1.8)
    ax[1,j].plot(TOBS,X_obs[j],"o",ms=5,color=INK,zorder=5)
    ax[1,j].set_xlabel("time (min)")
ax[0,0].set_ylabel("CP (mg L$^{-1}$)"); ax[1,0].set_ylabel("biomass (gDCW L$^{-1}$)")
plt.tight_layout(); plt.show()

In [ ]:
# harvest-time optimisation: a design question that the 5 h IPTG horizon could not address
Yv = Pp*1e-6*M_CP*1e3*PHI_CYT*Xp*1e3                     # mg/L
qy = np.percentile(Yv[:,2,:],[5,50,95],axis=0)
k  = int(np.argmax(qy[1]))
fig,ax=plt.subplots(figsize=(6.2,3.6))
ax.fill_between(t_fine,qy[0],qy[2],alpha=.28,color=MID)
ax.plot(t_fine,qy[1],color=MID,lw=2.2)
ax.axvline(t_fine[k],color=INK,ls="--",lw=1.3)
ax.axvline(300,color=STEEL,ls=":",lw=1.3)
ax.text(310,qy[1].max()*0.25,"old 300 min\nhorizon",fontsize=7,color=MID)
ax.set(xlabel="harvest time (min)",ylabel="CP yield (mg/L)",
       title="When to harvest (5.5 mM lactose)")
plt.tight_layout(); plt.show()
print(f"Predicted optimal harvest: {t_fine[k]:.0f} min ({t_fine[k]/60:.1f} h), "
      f"{qy[1][k]:.0f} mg/L (90% CI {qy[0][k]:.0f}-{qy[2][k]:.0f})")
print(f"Yield if harvested at 300 min: {np.interp(300,t_fine,qy[1]):.0f} mg/L")

---
## 10 · Held-out validation

The same scheme as in notebook 01: the model is refitted on three doses and used to predict the
fourth.

In [ ]:
HOLDOUT = 2
keep = np.array([i for i in range(len(DOSES)) if i != HOLDOUT])
_D,_X,_CP,_L,_G = DOSES,X_obs,CP_obs,LOG_CP,GEL_ID
DOSES,X_obs,CP_obs,LOG_CP,GEL_ID = _D[keep],_X[keep],_CP[keep],_L[keep],_G[keep]
model_ho = build_model()
with model_ho:
    idata_ho = pm.sample_smc(draws=max(DRAWS//2,200), chains=CHAINS,
                             random_seed=99, progressbar=False)
DOSES,X_obs,CP_obs,LOG_CP,GEL_ID = _D,_X,_CP,_L,_G
print(f"Refit without {_D[HOLDOUT]:g} mM; now predicting it.")

In [ ]:
TH_H = post_draws(idata_ho,250); Ph,Xh=[],[]
for d in TH_H:
    dd=dict(d); dd.update(FIXED)
    o=simulate(theta_full(dd),np.array([_D[HOLDOUT]]),t_fine)
    if o is not None: Ph.append(uM_to_mgL(o["P"][0])); Xh.append(o["X"][0])
Ph,Xh=np.array(Ph),np.array(Xh)
ah=np.exp(idata_ho.posterior["log_alpha"].values.reshape(-1,max(N_GEL,1))).mean(0)
au=ah[min(_G[HOLDOUT],ah.size-1)]

fig,ax=plt.subplots(1,2,figsize=(9,3.4))
q=np.percentile(Ph*au,[3,25,50,75,97],axis=0)
ax[0].fill_between(t_fine,q[0],q[4],alpha=.16,color=MID,label="94% predictive")
ax[0].fill_between(t_fine,q[1],q[3],alpha=.30,color=MID)
ax[0].plot(t_fine,q[2],color=MID,lw=2,label="median")
ax[0].plot(TOBS,_CP[HOLDOUT],"o",ms=7,color=INK,zorder=5,label="held-out data")
ax[0].set(xlabel="time (min)",ylabel="CP (mg L$^{-1}$)",
          title=f"Held-out: {_D[HOLDOUT]:g} mM (never fitted)"); ax[0].legend(fontsize=7)
qo=np.percentile(Xh,[3,50,97],axis=0)
ax[1].fill_between(t_fine,qo[0],qo[2],alpha=.2,color=SKY)
ax[1].plot(t_fine,qo[1],color=SKY,lw=2)
ax[1].plot(TOBS,_X[HOLDOUT],"o",ms=7,color=INK,zorder=5)
ax[1].set(xlabel="time (min)",ylabel="biomass (gDCW L$^{-1}$)",title="Held-out growth")
plt.tight_layout(); plt.show()

lo=np.interp(TOBS,t_fine,np.percentile(Ph*au,3,axis=0))
hi=np.interp(TOBS,t_fine,np.percentile(Ph*au,97,axis=0))
ins=((_CP[HOLDOUT]>=lo)&(_CP[HOLDOUT]<=hi)); cov=ins.sum()/ins.size
print(f"Inside 94% predictive interval: {ins.sum()}/{ins.size} ({100*cov:.0f}%, nominal 94%)")
if FAST_MODE:
    print("NOTE: FAST_MODE on -- under-sampled posteriors give unreliable coverage.")
    print("      Re-run with FAST_MODE=False before drawing any conclusion from this number.")
elif cov < 0.7:
    print("Well below nominal -> overconfident on unseen conditions.")
    print("Widen priors or add a Kennedy-O'Hagan discrepancy term before trusting forecasts.")
else:
    print("Coverage consistent with nominal: the held-out forecast is substantiated.")

---
## 11 · Export for Module M2

In [ ]:
draws = pd.DataFrame({v: idata.posterior[v].values.ravel() for v in FIT})
draws.to_csv("M1_lactose_posterior_draws.csv", index=False)
logd = np.log(draws)
pd.DataFrame({"median":draws.median(),"log_mean":logd.mean(),"log_sd":logd.std(),
              "q05":draws.quantile(.05),"q95":draws.quantile(.95)}
            ).to_csv("M1_lactose_posterior_moments.csv")
try:
    idata.to_netcdf("M1_lactose_idata.nc")
    print("wrote draws / moments / idata.nc\n")
except Exception as e:
    print(f"(NetCDF skipped: {type(e).__name__}; pip install h5netcdf for the full object)")
    print("wrote draws / moments\n")

cm = logd.corr(); print("Posterior correlations |r|>0.4:")
for i in range(len(FIT)):
    for j in range(i+1,len(FIT)):
        if abs(cm.iloc[i,j])>0.4: print(f"  {FIT[i]:9s} <-> {FIT[j]:9s}  r={cm.iloc[i,j]:+.2f}")
print("\nCarry correlated pairs into M2 as JOINT draws, never independent marginals.")

# re-run the bistability test across the posterior, as §4.3 requires
gaps=[]
for d in post_draws(idata,60):
    dd=dict(d); dd.update(FIXED); th=theta_full(dd)
    n=scan.size
    Y0n=np.concatenate([np.full(n,0.05),np.full(n,2.5),scan]+[np.zeros(n)]*7)
    Y0p=np.concatenate([np.full(n,0.05),np.full(n,2.5),scan,np.zeros(n),np.zeros(n),
                        np.ones(n),np.ones(n),np.zeros(n),np.zeros(n),np.zeros(n)])
    try:
        sn=solve_ivp(_rhs,(0,1440.),Y0n,t_eval=[1440.],args=(th,scan),method="LSODA",rtol=1e-6,atol=1e-9)
        sp=solve_ivp(_rhs,(0,1440.),Y0p,t_eval=[1440.],args=(th,scan),method="LSODA",rtol=1e-6,atol=1e-9)
        if sn.success and sp.success:
            a_=sn.y[9*n:10*n,-1]; b_=sp.y[9*n:10*n,-1]
            gaps.append(np.abs(b_-a_).max()/max(a_.max(),1e-9))
    except Exception: pass
gaps=np.array(gaps)
print(f"\nBistability across posterior: {(gaps>0.05).mean()*100:.0f}% of draws show >5% path dependence")
print("This fraction, not the single-parameter result of §4.3, is the quantity to report.")

---
## 12 · Next steps

**Before real data.** §8 should recover at least 10 of the 13 parameters, with all r̂ ≤ 1.01.

**When data arrive.** Set `USE_SYNTHETIC = False`, point §5 at the CSV file, and rerun.

**Design changes supported by this notebook:**

- **Residual glucose and lactose** should be assayed at 3–4 time points. The time of glucose
  exhaustion alone constrains `K_cat` and `q_Glc`, and it is the least expensive identifiability
  gain available.
- **Sampling should extend past 12 h.** Half-maximal accumulation occurs at 11–13 h; a run ending
  at 8 h cannot identify `k_deg` and will bias `G`.
- **An uninduced (no-lactose) arm should be carried for the full 24 h.** This is required: without
  it `eps` and `G` are correlated at r ≈ −0.98, and the ridge propagates into every yield forecast.
- **Gel identity should be recorded** for every lane.

**Choice of inducer.** IPTG and lactose are not interchangeable:

| | IPTG | Lactose / allolactose |
|---|---|---|
| Inducer available | Within minutes | Only after glucose is spent and LacY is built |
| Horizon | ~5 h | ~24 h |
| States / fitted parameters | 5 / 11 | 10 / 13 |
| Bimodality risk | Real in *lacY*⁺; avoidable with Tuner(DE3) | Damped by allolactose hydrolysis (test in §4.3) |
| Cost at scale | High | Low |
| Regulatory profile | Synthetic gratuitous inducer | Food-grade sugar |

For a veterinary vaccine intended for low-resource deployment, the last two rows are part of the
platform argument and not incidental.
---
## 13 · References

New in this notebook:

1. Jacob F, Monod J. Genetic regulatory mechanisms in the synthesis of proteins. *J Mol Biol.* 1961;3:318–356.
2. Novick A, Weiner M. Enzyme induction as an all-or-none phenomenon. *PNAS.* 1957;43(7):553–566.
3. Silverstone AE, Arditti RR, Magasanik B. Catabolite-insensitive revertants of lac promoter mutants. *PNAS.* 1970;66(3):773–779.
4. Jobe A, Bourgeois S. lac Repressor–operator interaction VI: the natural inducer of the lac operon. *J Mol Biol.* 1972;69(3):397–408.
5. Huber RE, Wallenfels K, Kurz G. The action of β-galactosidase (*E. coli*) on allolactose. *Can J Biochem.* 1975;53(9):1035–1038.
6. Huber RE, Kurz G, Wallenfels K. A quantitation of the factors which affect the hydrolase and transgalactosylase activities of β-galactosidase (*E. coli*) on lactose. *Biochemistry.* 1976;15(9):1994–2001.
7. Studier FW. Protein production by auto-induction in high-density shaking cultures. *Protein Expr Purif.* 2005;41(1):207–234.
8. Görke B, Stülke J. Carbon catabolite repression in bacteria: many ways to make the most out of nutrients. *Nat Rev Microbiol.* 2008;6(8):613–624.
9. Deutscher J. The mechanisms of carbon catabolite repression in bacteria. *Curr Opin Microbiol.* 2008;11(2):87–93.
10. Juers DH, Matthews BW, Huber RE. LacZ β-galactosidase: structure and function of an enzyme of historical and molecular biological importance. *Protein Sci.* 2012;21(12):1792–1807.
11. Wheatley RW, Lo S, Jancewicz LJ, Dugdale ML, Huber RE. Structural explanation for allolactose (lac operon inducer) synthesis by lacZ β-galactosidase. *J Biol Chem.* 2013;288(18):12993–13005.
12. Santillán M, Mackey MC, Zeron ES. Origin of bistability in the lac operon. *Biophys J.* 2007;92(11):3830–3842.

Carried over: Scott 2010; Klumpp 2009; Bremer & Dennis 2008; Marbach & Bettenbrock 2012;
Ozbudak 2004; Lewis 1996; Yildirim & Mackey 2003; Studier & Moffatt 1986; Bernstein 2002;
Rocke & Durbin 2001; Gelman & Hill 2007; Kennedy & O'Hagan 2001; Del Moral 2006; Raue 2009.